In [ ]:
#!/usr/bin/env python3
"""
Multi-scale solar-filament oscillation detector using CNN + Conformal Prediction.

This version removes the histogram-based "mode band" selection and replaces it
with a simpler reporting pipeline:

  1) At each degraded scale N:
       - compute PSD per degraded pixel
       - compute pixel-specific tau_i(f)
       - keep significant peaks where PSD > tau_i
       - cluster significant periods directly by tolerance
       - for each clustered period, run connected components in the image plane

  2) Save one record per connected component:
       - scale N / stride S
       - period group center
       - component centroid / bbox / area / strength

  3) Produce:
       - per-scale detection tables
       - global detection table
       - period-vs-scale scatter plot
       - spatial plots for each period family across scales
"""

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import csv
import math
import json
from collections import defaultdict

import numpy as np
import h5py as h5
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
from joblib import Parallel, delayed, parallel_backend

from skimage.measure import label, regionprops
from scipy.ndimage import uniform_filter, binary_dilation
from scipy.signal import find_peaks

import tensorflow as tf
from tensorflow.keras import Model, layers
from sklearn.preprocessing import MinMaxScaler
from astropy.timeseries import LombScargle


# ============================================================
# PARALLELISM
# ============================================================
N_FILAMENT_WORKERS = 64
N_PIXEL_WORKERS = 64

# ============================================================
# CNN FREQUENCY GRID
# ============================================================
CNN_FREQUENCY_GRID = np.linspace(1 / (1440 * 60), 1 / 120, 2000).astype(np.float32)

# Approx. 5-sigma upper tail
CP_DELTA = 0.0000006

# ============================================================
# DETECTION BAND
# ============================================================
DETECTION_FMIN_HZ = 1.0 / (3.0 * 3600.0)   # 3 hours
DETECTION_FMAX_HZ = 1.0e-3                 # 1 mHz = 16.67 min


# ============================================================
# CNN noise model
# ============================================================
def noise_model(x, a, alpha, b):
    return a * x ** (-alpha) + b


class CNN(Model):
    def __init__(self):
        super().__init__()
        self.original_dim = 2000
        self.dropout_rate = 0.2

        self.conv_block = tf.keras.Sequential([
            layers.Input(shape=(self.original_dim, 1)),

            layers.Conv1D(64, 5, activation="relu", padding="valid"),
            layers.Conv1D(64, 8, activation="relu", padding="valid"),
            layers.MaxPooling1D(2),

            layers.Conv1D(64, 11, activation="relu", padding="valid"),
            layers.Conv1D(64, 10, activation="relu", padding="valid"),
            layers.MaxPooling1D(2),

            layers.Conv1D(64, 11, activation="relu", padding="valid"),
            layers.Conv1D(64, 10, activation="relu", padding="valid"),
            layers.MaxPooling1D(2),

            layers.Conv1D(64, 11, activation="relu", padding="valid"),
            layers.Conv1D(64, 10, activation="relu", padding="valid"),
            layers.MaxPooling1D(2),

            layers.Flatten(),

            layers.Dense(1024, activation="relu"),
            layers.Dropout(self.dropout_rate),

            layers.Dense(512, activation="relu"),
            layers.Dropout(self.dropout_rate),

            layers.Dense(256, activation="relu"),
            layers.Dropout(self.dropout_rate),

            layers.Dense(3, activation="sigmoid"),
        ])

    def call(self, input_features):
        x = tf.expand_dims(input_features, -1)
        return self.conv_block(x)


def load_cnn(weights_path: str) -> CNN:
    model = CNN()
    model.load_weights(weights_path)
    return model


def _build_scaler() -> MinMaxScaler:
    sc = MinMaxScaler()
    sc.min_ = np.array([1.5661751, 0.19527236, 2.219622], dtype=np.float32)
    sc.scale_ = np.array([0.14172548, 2.0891168, 0.396035], dtype=np.float32)
    return sc


def cnn_predict_noise_params(model: CNN, psds: np.ndarray, scaler: MinMaxScaler) -> np.ndarray:
    raw = model.predict(psds, verbose=0)
    raw = np.asarray(raw, np.float64)

    if raw.ndim != 2 or raw.shape[1] != 3:
        raise RuntimeError(f"CNN returned unexpected shape: {raw.shape}")
    if not np.all(np.isfinite(raw)):
        raise RuntimeError("CNN prediction contains non-finite values.")

    rescaled = scaler.inverse_transform(raw)
    if not np.all(np.isfinite(rescaled)):
        raise RuntimeError("Scaler inverse_transform produced non-finite values.")

    params = (10.0 ** rescaled).astype(np.float64)
    if not np.all(np.isfinite(params)):
        raise RuntimeError("Noise parameters contain non-finite values after exponentiation.")

    return params


def compute_ls_psd_safe(tt, yy, freqs, *, apply_hann=True, min_var=1e-12, min_energy=1e-20):
    tt = np.asarray(tt, np.float64)
    yy = np.asarray(yy, np.float64)

    ok = np.isfinite(tt) & np.isfinite(yy)
    if ok.sum() < 10:
        return None

    tt = tt[ok]
    yy = yy[ok]

    if yy.size < 10:
        return None

    yy = yy - np.mean(yy)

    var = np.var(yy)
    if not np.isfinite(var) or var <= float(min_var):
        return None

    if apply_hann and yy.size >= 8:
        yy = yy * np.hanning(yy.size)

    energy = np.sum(yy * yy)
    if not np.isfinite(energy) or energy <= float(min_energy):
        return None

    pxx = LombScargle(tt, yy).power(freqs)
    pxx = np.asarray(pxx, np.float64)

    if pxx.shape != freqs.shape:
        return None
    if not np.all(np.isfinite(pxx)):
        return None
    if np.all(pxx <= 0):
        return None

    return pxx.astype(np.float32)


# ============================================================
# CONFORMAL PREDICTION CALIBRATION
# ============================================================
def compute_cp_calibration(
    images: np.ndarray,
    tdeltas: np.ndarray,
    masks: np.ndarray,
    cnn_model: CNN,
    scaler: MinMaxScaler,
    *,
    n_calib: int = 100_000,
    delta: float = 0.001,
    rng_seed: int = 42,
    n_jobs: int = N_PIXEL_WORKERS,
    batch_size_cnn: int = 4096,
):
    T, H, W = images.shape
    freqs = CNN_FREQUENCY_GRID.astype(np.float64)
    rng = np.random.default_rng(int(rng_seed))
    eps = 1e-30

    mask0 = (masks[0] > 0)
    exclusion_mask = binary_dilation(mask0, iterations=10)
    calib_candidate_mask = ~exclusion_mask

    calib_candidate_mask &= (images[0] > 0)

    finite_count = np.isfinite(images).sum(axis=0)
    valid_mask = calib_candidate_mask & (finite_count >= max(10, int(0.8 * T)))

    temporal_mean = np.nanmean(images.astype(np.float64), axis=0)
    mean_vals = temporal_mean[valid_mask & np.isfinite(temporal_mean)]
    if mean_vals.size == 0:
        raise RuntimeError("No valid candidate pixels found for CP calibration.")

    p5, p95 = np.percentile(mean_vals, [5.0, 95.0])
    valid_mask &= np.isfinite(temporal_mean) & (temporal_mean >= p5) & (temporal_mean <= p95)

    ys, xs = np.where(valid_mask)
    n_avail = len(ys)
    if n_avail == 0:
        raise RuntimeError("No valid calibration pixels found after filtering.")

    n_draw = min(int(n_calib), n_avail)
    if n_draw < n_avail:
        idx = rng.choice(n_avail, size=n_draw, replace=False)
        ys = ys[idx]
        xs = xs[idx]

    print(
        f"[CP calib] Candidate mask built from ~dilate(masks[0], 10). "
        f"Temporal-mean filter kept [{p5:.3e}, {p95:.3e}] percentile range."
    )
    print(
        f"[CP calib] Drawing {n_draw} pixels for calibration "
        f"(delta={delta}, -> {100*(1-delta):.5f}% upper quantile)."
    )

    t_ok = np.isfinite(tdeltas)
    tdeltas_clean = tdeltas[t_ok]

    def _psd_one(y_px, x_px):
        y = images[t_ok, y_px, x_px].astype(np.float64)
        ok = np.isfinite(y)
        if ok.sum() < 10:
            return None

        yy = y[ok]
        tt = tdeltas_clean[ok]

        return compute_ls_psd_safe(
            tt, yy, freqs,
            apply_hann=True,
            min_var=1e-12,
            min_energy=1e-20,
        )

    print("[CP calib] Computing calibration PSDs ...")
    psds_list = Parallel(n_jobs=n_jobs)(
        delayed(_psd_one)(int(ys[k]), int(xs[k]))
        for k in tqdm(range(n_draw), desc="Calib PSDs", leave=False)
    )
    psds_list = [p for p in psds_list if p is not None]

    if not psds_list:
        raise RuntimeError("All sampled calibration pixels were invalid for PSD computation.")

    calib_psds = np.vstack(psds_list).astype(np.float32)
    row_ok = np.all(np.isfinite(calib_psds), axis=1) & np.any(calib_psds > 0, axis=1)
    calib_psds = calib_psds[row_ok]

    n_eff = calib_psds.shape[0]
    if n_eff == 0:
        raise RuntimeError("No valid finite calibration PSDs remained after filtering.")

    print(f"[CP calib] Effective calibration PSDs after finite filtering: {n_eff}")

    print("[CP calib] Running CNN inference ...")
    n_batches = math.ceil(n_eff / batch_size_cnn)
    all_params = []
    for b in range(n_batches):
        sl = slice(b * batch_size_cnn, (b + 1) * batch_size_cnn)
        all_params.append(cnn_predict_noise_params(cnn_model, calib_psds[sl], scaler))
    noise_params = np.vstack(all_params)

    print("[CP calib] Building residual statistics ...")
    R = np.empty_like(calib_psds, dtype=np.float64)

    for i in range(n_eff):
        Shat = noise_model(freqs, *noise_params[i])
        psd_safe = np.maximum(calib_psds[i].astype(np.float64), eps)
        shat_safe = np.maximum(Shat, eps)
        R[i, :] = np.log(psd_safe) - np.log(shat_safe)

    row_ok = np.all(np.isfinite(R), axis=1)
    R = R[row_ok]
    if R.shape[0] == 0:
        raise RuntimeError("All calibration residual rows were non-finite.")

    m = np.median(R, axis=0)
    s = 1.4826 * np.median(np.abs(R - m[None, :]), axis=0)
    s = np.maximum(s, 1e-8)

    Z = (R - m[None, :]) / s[None, :]
    qk = np.quantile(Z, 1.0 - delta, axis=0, method="higher")

    if not (np.all(np.isfinite(m)) and np.all(np.isfinite(s)) and np.all(np.isfinite(qk))):
        raise RuntimeError("CP statistics contain non-finite values; aborting cache write.")

    factor = np.exp(m + qk * s)
    print(
        f"[CP calib] Done. multiplicative factor range: "
        f"[{factor.min():.3e}, {factor.max():.3e}]"
    )

    return m.astype(np.float64), s.astype(np.float64), qk.astype(np.float64)


def get_cp_cache_path(day_outroot, day, cp_delta):
    return os.path.join(
        os.path.abspath(day_outroot),
        f"cp_stats_{day}_delta{cp_delta:.6f}_dil10_p05p95.npz"
    )


def ensure_cp_cache_for_day(
    *,
    day,
    data_h5,
    masks_h5,
    outroot,
    cnn_weights_path,
    cp_n_calib=100_000,
    cp_delta=0.01,
    cp_batch_size_cnn=4096,
    n_jobs=N_PIXEL_WORKERS,
):
    os.makedirs(outroot, exist_ok=True)

    cp_cache_path = get_cp_cache_path(outroot, day, cp_delta)
    if os.path.exists(cp_cache_path):
        print(f"[CP precompute] Cache already exists: {cp_cache_path}")
        return cp_cache_path

    print(f"[CP precompute] Building daily CP cache for day={day}")

    with h5.File(data_h5, "r") as hf:
        images = np.array(hf["time_series"][:], dtype=np.float32)
        tdeltas = np.array(hf["tdeltas"][:], dtype=np.float64)

    with h5.File(masks_h5, "r") as hm:
        masks = np.array(hm["masks"][:], dtype=np.uint8)

    order = np.argsort(tdeltas)
    tdeltas = tdeltas[order]
    images = images[order]
    masks = masks[order]

    cnn_model = load_cnn(cnn_weights_path)
    scaler = _build_scaler()

    m_cp, s_cp, qk = compute_cp_calibration(
        images, tdeltas, masks, cnn_model, scaler,
        n_calib=int(cp_n_calib),
        delta=float(cp_delta),
        n_jobs=int(n_jobs),
        batch_size_cnn=int(cp_batch_size_cnn),
    )

    np.savez(
        cp_cache_path,
        m=m_cp,
        s=s_cp,
        qk=qk,
        delta=np.float64(cp_delta),
    )

    print(f"[CP precompute] Done -> {cp_cache_path}")
    return cp_cache_path


# ============================================================
# IMAGE DEGRADATION
# ============================================================
def block_reduce_mean(frame2d, N, S):
    H, W = frame2d.shape
    row_starts = np.arange(0, H - N + 1, S, dtype=np.intp)
    col_starts = np.arange(0, W - N + 1, S, dtype=np.intp)

    if S == N:
        rh = (H // N) * N
        rw = (W // N) * N
        return frame2d[:rh, :rw].reshape(H // N, N, W // N, N).mean(axis=(1, 3))

    out = np.empty((len(row_starts), len(col_starts)), dtype=np.float32)
    for ii, r0 in enumerate(row_starts):
        for jj, c0 in enumerate(col_starts):
            out[ii, jj] = frame2d[r0:r0+N, c0:c0+N].mean()
    return out


def degrade_stack_unweighted(images, N, S):
    T, H, W = images.shape
    row_starts = np.arange(0, H - N + 1, S, dtype=np.intp)
    col_starts = np.arange(0, W - N + 1, S, dtype=np.intp)
    oh = len(row_starts)
    ow = len(col_starts)

    if S == N:
        rh = oh * N
        rw = ow * N
        return images[:, :rh, :rw].reshape(T, oh, N, ow, N).mean(axis=(2, 4)).astype(np.float32)

    out = np.empty((T, oh, ow), dtype=np.float32)
    for k in range(T):
        f = images[k]
        for ii, r0 in enumerate(row_starts):
            for jj, c0 in enumerate(col_starts):
                out[k, ii, jj] = f[r0:r0+N, c0:c0+N].mean()
    return out


def degrade_mask_coverage(mask2d, N, S):
    m = (mask2d > 0).astype(np.float32)
    return block_reduce_mean(m, N, S)


# ============================================================
# HELPERS
# ============================================================
def weighted_median_log(logx, w):
    logx = np.asarray(logx, np.float64)
    w = np.asarray(w, np.float64)

    m = np.isfinite(logx) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan

    logx = logx[m]
    w = w[m]
    order = np.argsort(logx)
    cdf = np.cumsum(w[order]) / np.sum(w)
    idx = np.searchsorted(cdf, 0.5)
    return float(logx[order][idx])


def local_nan_std(x, win=3):
    x = np.asarray(x, np.float32)
    m = np.isfinite(x).astype(np.float32)
    x0 = np.nan_to_num(x, nan=0.0)
    n = uniform_filter(m, size=win, mode="nearest")
    s1 = uniform_filter(x0 * m, size=win, mode="nearest")
    s2 = uniform_filter(x0**2 * m, size=win, mode="nearest")
    mean = s1 / np.maximum(n, 1e-9)
    var = np.maximum(s2 / np.maximum(n, 1e-9) - mean * mean, 0.0)
    std = np.sqrt(var)
    std[n < 1e-6] = np.nan
    return std.astype(np.float32)


def list_candidate_regions_from_mask(mask0, *, min_area=2000):
    lbl = label((mask0 > 0).astype(np.uint8))
    regs = [r for r in regionprops(lbl) if r.area >= int(min_area)]
    return sorted(regs, key=lambda x: x.area, reverse=True)


def select_roi_bbox_from_first_mask(mask0, *, min_area=2000, pick="largest", pick_index=0):
    regs = list_candidate_regions_from_mask(mask0, min_area=int(min_area))
    if not regs:
        raise RuntimeError("No regions found above min_area")

    if pick == "largest":
        return regs[0].bbox

    pick_index = int(pick_index)
    if pick_index >= len(regs):
        raise RuntimeError(f"pick_index {pick_index} out of range for {len(regs)} regions")
    return regs[pick_index].bbox


def expand_bbox(bbox, pad, H, W):
    minr, minc, maxr, maxc = bbox
    return (max(0, minr - pad), max(0, minc - pad), min(H, maxr + pad), min(W, maxc + pad))


def degraded_center_to_roi_xy(i, j, N, S):
    return float(j * S + 0.5 * N), float(i * S + 0.5 * N)


def patch_mask_to_roi(patch_mask, N, S, roi_H, roi_W):
    roi = np.zeros((roi_H, roi_W), dtype=bool)
    if patch_mask is None or not np.any(patch_mask):
        return roi

    rr, cc = np.where(patch_mask)
    if rr.size == 0:
        return roi

    y0s = (rr * S).astype(np.intp)
    x0s = (cc * S).astype(np.intp)
    y1s = np.minimum(y0s + N, roi_H)
    x1s = np.minimum(x0s + N, roi_W)

    for y0, y1, x0, x1 in zip(y0s, y1s, x0s, x1s):
        roi[y0:y1, x0:x1] = True
    return roi


def bbox_from_mask(mask_bool):
    mask_bool = np.asarray(mask_bool, bool)
    rr, cc = np.where(mask_bool)
    if rr.size == 0:
        return None
    return (int(rr.min()), int(cc.min()), int(rr.max() + 1), int(cc.max() + 1))


def choose_scales_distinct_quotients(bbox_H, start=50, stop=2):
    prev = None
    vals = []
    for N in range(int(start), int(stop), -1):
        q = int(bbox_H // N)
        if prev is None or q != prev:
            vals.append(int(N))
            prev = q
    return vals


def choose_scales_log_ladder(Nmax, Nmin, n=12):
    vals = np.unique(np.round(np.geomspace(int(Nmax), int(Nmin), int(n))).astype(int))[::-1]
    return [int(v) for v in vals if v >= int(Nmin)]


def apply_null_time_transform(stack, mode, rng):
    if mode == "none":
        return stack
    if mode == "circular_shift":
        return np.roll(stack, shift=int(rng.integers(1, stack.shape[0])), axis=0)
    if mode == "shuffle":
        return stack[rng.permutation(stack.shape[0])]
    raise ValueError("null mode must be: none | circular_shift | shuffle")


# ============================================================
# MULTI-PEAK ANALYSIS WITH PIXEL-SPECIFIC TAU
# ============================================================
def analyze_degraded_stack_multipeak_cp(
    t,
    stack,
    cnn_model,
    scaler,
    m,
    s,
    qk,
    *,
    keep_mask=None,
    min_finite_frac=0.8,
    apply_hann=True,
    top_m_peaks=4,
    peak_min_prom_frac=0.05,
    n_jobs=N_PIXEL_WORKERS,
    batch_size_cnn=4096,
    detect_fmin_hz=DETECTION_FMIN_HZ,
    detect_fmax_hz=DETECTION_FMAX_HZ,
):
    freqs = CNN_FREQUENCY_GRID.astype(np.float64)
    detect_fmin_hz = float(detect_fmin_hz)
    detect_fmax_hz = float(detect_fmax_hz)

    if not (np.isfinite(detect_fmin_hz) and np.isfinite(detect_fmax_hz) and detect_fmin_hz < detect_fmax_hz):
        raise ValueError("Invalid detection band.")

    detect_band = (freqs >= detect_fmin_hz) & (freqs <= detect_fmax_hz)
    if not np.any(detect_band):
        raise RuntimeError("Detection band does not overlap CNN_FREQUENCY_GRID.")

    t = np.asarray(t, np.float64)
    m = np.asarray(m, np.float64)
    s = np.asarray(s, np.float64)
    qk = np.asarray(qk, np.float64)

    if not (len(freqs) == len(m) == len(s) == len(qk)):
        raise ValueError("freqs, m, s, qk must all have the same length.")

    tau_factor = np.exp(m + qk * s).astype(np.float64)

    T, h, w = stack.shape
    M = int(top_m_peaks)

    periods_m = np.full((h, w, M), np.nan, dtype=np.float32)
    freqs_m = np.full((h, w, M), np.nan, dtype=np.float32)
    z_m = np.full((h, w, M), np.nan, dtype=np.float32)
    best_period_min = np.full((h, w), np.nan, dtype=np.float32)
    best_z = np.full((h, w), np.nan, dtype=np.float32)
    sig_any = np.zeros((h, w), dtype=bool)

    if keep_mask is None:
        keep_mask = np.ones((h, w), dtype=bool)
    else:
        keep_mask = keep_mask.astype(bool)

    t_ok_base = np.isfinite(t)
    min_pts = max(5, int(min_finite_frac * T))

    def _one_pixel_psd(idx_flat):
        i = idx_flat // w
        j = idx_flat % w

        if not keep_mask[i, j]:
            return i, j, None

        y = stack[:, i, j].astype(np.float64)
        ok = t_ok_base & np.isfinite(y)
        if ok.sum() < min_pts:
            return i, j, None

        tt = t[ok]
        yy = y[ok]

        pxx = compute_ls_psd_safe(
            tt, yy, freqs,
            apply_hann=apply_hann,
            min_var=1e-12,
            min_energy=1e-20,
        )
        return i, j, pxx

    flat_ids = np.flatnonzero(keep_mask)
    with parallel_backend("loky", n_jobs=n_jobs):
        results = Parallel()(
            delayed(_one_pixel_psd)(k)
            for k in tqdm(flat_ids, desc="LS per degraded pixel (CP)", leave=False)
        )

    valid_results = [(i, j, pxx) for (i, j, pxx) in results if pxx is not None]
    if not valid_results:
        logp = np.full((h, w), np.nan, np.float32)
        coherence_std = local_nan_std(logp, win=3)
        patch_score = uniform_filter(np.nan_to_num(best_z, nan=0.0), size=3, mode="nearest").astype(np.float32)

        return dict(
            periods_m=periods_m,
            freqs_m=freqs_m,
            z_m=z_m,
            best_period_min=best_period_min,
            best_z=best_z,
            sig_any=sig_any,
            keep_mask=keep_mask,
            coherence_logp_std=coherence_std,
            patch_score=patch_score,
        )

    coords = [(i, j) for (i, j, _) in valid_results]
    psds = np.vstack([pxx for (_, _, pxx) in valid_results]).astype(np.float32)

    n_pix = psds.shape[0]
    n_batches = math.ceil(n_pix / batch_size_cnn)
    all_params = []
    for b in range(n_batches):
        sl = slice(b * batch_size_cnn, (b + 1) * batch_size_cnn)
        all_params.append(cnn_predict_noise_params(cnn_model, psds[sl], scaler))
    noise_params = np.vstack(all_params)

    for (i, j), pxx32, params in zip(coords, psds, noise_params):
        pxx = pxx32.astype(np.float64)
        shat = noise_model(freqs, *params)
        tau = np.maximum(shat, 1e-30) * tau_factor

        sigmask = detect_band & np.isfinite(pxx) & np.isfinite(tau) & (pxx > tau)
        if not np.any(sigmask):
            continue

        snr = pxx / (tau + 1e-300)
        snr_sig = np.where(sigmask, snr, -np.inf)
        finite_sig = np.isfinite(snr_sig)
        if not np.any(finite_sig):
            continue

        snr_max = np.max(snr_sig[finite_sig])
        if not np.isfinite(snr_max):
            continue

        prom = float(peak_min_prom_frac) * float(snr_max)
        peaks, _ = find_peaks(np.nan_to_num(snr_sig, nan=-np.inf), prominence=prom)

        if peaks.size == 0:
            cand = np.argsort(snr_sig)[::-1]
            peaks = cand[np.isfinite(snr_sig[cand])][:M]

        peaks = np.asarray(peaks, int)
        peaks = peaks[np.isfinite(snr_sig[peaks])]
        if peaks.size == 0:
            continue

        order = peaks[np.argsort(snr_sig[peaks])[::-1]][:M]

        sig_any[i, j] = True
        zz = snr[order].astype(np.float32)

        for m_idx, k in enumerate(order):
            bf = float(freqs[int(k)])
            bpmin = float((1.0 / bf) / 60.0)
            periods_m[i, j, m_idx] = bpmin
            freqs_m[i, j, m_idx] = bf
            z_m[i, j, m_idx] = float(zz[m_idx])

        best_period_min[i, j] = float(periods_m[i, j, 0])
        best_z[i, j] = float(z_m[i, j, 0])

    logp = np.full((h, w), np.nan, np.float32)
    m_ok = sig_any & np.isfinite(best_period_min) & (best_period_min > 0)
    logp[m_ok] = np.log(best_period_min[m_ok])

    coherence_std = local_nan_std(logp, win=3)
    patch_score = uniform_filter(np.nan_to_num(best_z, nan=0.0), size=3, mode="nearest").astype(np.float32)

    return dict(
        periods_m=periods_m,
        freqs_m=freqs_m,
        z_m=z_m,
        best_period_min=best_period_min,
        best_z=best_z,
        sig_any=sig_any,
        keep_mask=keep_mask,
        coherence_logp_std=coherence_std,
        patch_score=patch_score,
    )


# ============================================================
# PERIOD GROUPING AND COMPONENT EXTRACTION
# ============================================================
def periods_match(Pa, Pb, *, period_tol_frac=0.15, period_abs_tol_min=2.0):
    Pa = float(Pa)
    Pb = float(Pb)
    if not (np.isfinite(Pa) and np.isfinite(Pb) and Pa > 0 and Pb > 0):
        return False

    if abs(Pa - Pb) <= float(period_abs_tol_min):
        return True

    return abs(math.log(Pa) - math.log(Pb)) <= math.log1p(float(period_tol_frac))


def cluster_period_values(
    periods,
    weights,
    *,
    period_tol_frac=0.15,
    period_abs_tol_min=2.0,
    max_groups=None,
):
    periods = np.asarray(periods, np.float64)
    weights = np.asarray(weights, np.float64)

    m = np.isfinite(periods) & (periods > 0) & np.isfinite(weights) & (weights > 0)
    if not m.any():
        return []

    p = periods[m]
    w = weights[m]
    order = np.argsort(w)[::-1]

    clusters = []

    def _recompute_center(periods_list, weights_list):
        pp = np.asarray(periods_list, np.float64)
        ww = np.asarray(weights_list, np.float64)
        lp_med = weighted_median_log(np.log(pp), ww)
        if np.isfinite(lp_med):
            return float(np.exp(lp_med))
        return float(np.nanmedian(pp))

    for idx in order:
        pi = float(p[idx])
        wi = float(w[idx])

        best_k = None
        best_dist = np.inf

        for k, cl in enumerate(clusters):
            c = float(cl["center"])
            if periods_match(
                pi, c,
                period_tol_frac=period_tol_frac,
                period_abs_tol_min=period_abs_tol_min,
            ):
                dist = abs(math.log(pi) - math.log(c))
                if dist < best_dist:
                    best_k = k
                    best_dist = dist

        if best_k is None:
            clusters.append(dict(
                periods=[pi],
                weights=[wi],
                center=pi,
                total_weight=wi,
                count=1,
            ))
        else:
            cl = clusters[best_k]
            cl["periods"].append(pi)
            cl["weights"].append(wi)
            cl["center"] = _recompute_center(cl["periods"], cl["weights"])
            cl["total_weight"] = float(np.sum(cl["weights"]))
            cl["count"] = int(len(cl["periods"]))

    merged = True
    while merged and len(clusters) > 1:
        merged = False
        clusters = sorted(clusters, key=lambda d: (d["total_weight"], d["count"]), reverse=True)
        used = np.zeros(len(clusters), dtype=bool)
        new_clusters = []

        for i in range(len(clusters)):
            if used[i]:
                continue

            base_p = list(clusters[i]["periods"])
            base_w = list(clusters[i]["weights"])
            base_center = float(clusters[i]["center"])
            used[i] = True

            for j in range(i + 1, len(clusters)):
                if used[j]:
                    continue

                cj = float(clusters[j]["center"])
                if periods_match(
                    base_center, cj,
                    period_tol_frac=period_tol_frac,
                    period_abs_tol_min=period_abs_tol_min,
                ):
                    base_p.extend(clusters[j]["periods"])
                    base_w.extend(clusters[j]["weights"])
                    used[j] = True
                    merged = True

            center = _recompute_center(base_p, base_w)
            new_clusters.append(dict(
                periods=base_p,
                weights=base_w,
                center=center,
                total_weight=float(np.sum(base_w)),
                count=int(len(base_p)),
            ))

        clusters = new_clusters

    clusters = sorted(clusters, key=lambda d: (d["total_weight"], d["count"]), reverse=True)
    if max_groups is not None:
        clusters = clusters[:int(max_groups)]
    return clusters


def period_group_maps(periods_m, z_m, keep_mask, sig_any, center_min, *, tol_frac, abs_tol_min):
    periods_m = np.asarray(periods_m, np.float32)
    z_m = np.asarray(z_m, np.float32)
    keep_mask = np.asarray(keep_mask, bool)
    sig_any = np.asarray(sig_any, bool)

    center = float(center_min)
    tol = float(tol_frac) * center

    valid = (
        sig_any[..., None] & keep_mask[..., None] &
        np.isfinite(periods_m) & (periods_m > 0) &
        np.isfinite(z_m) & (z_m > 0)
    )

    in_group = valid & ((np.abs(periods_m - center) <= tol) | (np.abs(periods_m - center) <= float(abs_tol_min)))
    z_group = np.where(in_group, z_m, np.float32(-np.inf))

    arg = np.argmax(z_group, axis=-1)
    score_map = z_group[
        np.arange(z_group.shape[0])[:, None],
        np.arange(z_group.shape[1])[None, :],
        arg
    ]
    group_mask = np.isfinite(score_map) & (score_map > -np.inf)

    period_map = np.take_along_axis(periods_m, arg[..., None], axis=-1)[..., 0]
    period_map = np.where(group_mask, period_map, np.nan).astype(np.float32)
    score_map = np.where(group_mask, score_map, np.nan).astype(np.float32)

    return group_mask.astype(bool), score_map, period_map


def build_component_record(
    *,
    day,
    roi_pick_index,
    detection_id,
    scale_idx,
    scale_name,
    N,
    S,
    period_group_id,
    period_group_center_min,
    component_idx,
    component_mask,
    score_map,
    period_map,
    roi_H,
    roi_W,
    roi_miny,
    roi_minx,
    H,
    W,
):
    rr, cc = np.where(component_mask)
    w = np.nan_to_num(score_map[rr, cc], nan=0.0, posinf=0.0, neginf=0.0)
    strength = float(w.sum())

    periods = period_map[rr, cc]
    m = np.isfinite(periods) & (periods > 0) & np.isfinite(w) & (w > 0)

    if m.sum() >= 1:
        p_comp = float(np.exp(weighted_median_log(np.log(periods[m].astype(np.float64)), w[m].astype(np.float64))))
        p_mean = float(np.average(periods[m].astype(np.float64), weights=w[m].astype(np.float64)))
    else:
        p_comp = np.nan
        p_mean = np.nan

    wsum = float(w.sum()) + 1e-12
    ci = float((w * rr).sum() / wsum)
    cj = float((w * cc).sum() / wsum)

    centroid_roi_x, centroid_roi_y = degraded_center_to_roi_xy(ci, cj, N, S)
    centroid_full_x = float(centroid_roi_x + roi_minx)
    centroid_full_y = float(centroid_roi_y + roi_miny)

    bbox_deg = bbox_from_mask(component_mask)
    if bbox_deg is None:
        bbox_deg = (None, None, None, None)

    roi_mask = patch_mask_to_roi(component_mask, N, S, roi_H, roi_W)
    bbox_roi = bbox_from_mask(roi_mask)
    if bbox_roi is None:
        bbox_roi = (None, None, None, None)

    if bbox_roi[0] is None:
        bbox_full = (None, None, None, None)
    else:
        bbox_full = (
            int(bbox_roi[0] + roi_miny),
            int(bbox_roi[1] + roi_minx),
            int(bbox_roi[2] + roi_miny),
            int(bbox_roi[3] + roi_minx),
        )

    return dict(
        detection_id=int(detection_id),
        day=str(day),
        roi_pick_index=int(roi_pick_index),
        scale_idx=int(scale_idx),
        scale_name=str(scale_name),
        N=int(N),
        S=int(S),
        period_group_id=int(period_group_id),
        period_group_center_min=float(period_group_center_min),
        component_idx=int(component_idx),
        period_min=float(p_comp) if np.isfinite(p_comp) else None,
        period_mean_min=float(p_mean) if np.isfinite(p_mean) else None,
        strength=float(strength),
        area_degpx=int(component_mask.sum()),
        centroid_deg_i=float(ci),
        centroid_deg_j=float(cj),
        centroid_roi_x=float(centroid_roi_x),
        centroid_roi_y=float(centroid_roi_y),
        centroid_full_x=float(centroid_full_x),
        centroid_full_y=float(centroid_full_y),
        bbox_deg_min_i=bbox_deg[0],
        bbox_deg_min_j=bbox_deg[1],
        bbox_deg_max_i=bbox_deg[2],
        bbox_deg_max_j=bbox_deg[3],
        bbox_roi_min_y=bbox_roi[0],
        bbox_roi_min_x=bbox_roi[1],
        bbox_roi_max_y=bbox_roi[2],
        bbox_roi_max_x=bbox_roi[3],
        bbox_full_min_y=bbox_full[0],
        bbox_full_min_x=bbox_full[1],
        bbox_full_max_y=bbox_full[2],
        bbox_full_max_x=bbox_full[3],
        roi_mask=roi_mask.astype(bool),
    )


def extract_scale_period_components(
    *,
    day,
    roi_pick_index,
    scale_idx,
    scale_name,
    N,
    S,
    maps,
    roi_H,
    roi_W,
    roi_miny,
    roi_minx,
    H,
    W,
    period_tol_frac=0.05,
    period_abs_tol_min=5.0,
    min_component_area=1,
    max_period_groups=None,
    max_components_per_group=None,
    base_detection_id=0,
):
    valid = (
        maps["sig_any"][..., None] & maps["keep_mask"][..., None] &
        np.isfinite(maps["periods_m"]) & (maps["periods_m"] > 0) &
        np.isfinite(maps["z_m"]) & (maps["z_m"] > 0)
    )

    periods = maps["periods_m"][valid].astype(np.float64)
    weights = maps["z_m"][valid].astype(np.float64)

    clusters = cluster_period_values(
        periods, weights,
        period_tol_frac=period_tol_frac,
        period_abs_tol_min=period_abs_tol_min,
        max_groups=max_period_groups,
    )

    detections = []
    group_summaries = []
    detection_id = int(base_detection_id)

    for gid, cl in enumerate(clusters):
        center_min = float(cl["center"])

        group_mask, score_map, period_map = period_group_maps(
            maps["periods_m"], maps["z_m"],
            maps["keep_mask"], maps["sig_any"],
            center_min,
            tol_frac=period_tol_frac,
            abs_tol_min=period_abs_tol_min,
        )

        if not np.any(group_mask):
            continue

        lbl = label(group_mask.astype(np.uint8), connectivity=2)
        props = regionprops(lbl)

        comp_infos = []
        for r in props:
            if r.area < int(min_component_area):
                continue

            comp_mask = (lbl == r.label)
            rr = r.coords[:, 0]
            cc = r.coords[:, 1]
            comp_strength = float(np.nansum(np.nan_to_num(score_map[rr, cc], nan=0.0)))
            if not np.isfinite(comp_strength) or comp_strength <= 0:
                continue

            comp_infos.append((comp_strength, comp_mask))

        comp_infos.sort(key=lambda x: x[0], reverse=True)
        if max_components_per_group is not None:
            comp_infos = comp_infos[:int(max_components_per_group)]

        if not comp_infos:
            continue

        group_summaries.append(dict(
            scale_idx=int(scale_idx),
            scale_name=str(scale_name),
            N=int(N),
            S=int(S),
            period_group_id=int(gid),
            period_group_center_min=float(center_min),
            n_pixels=int(group_mask.sum()),
            n_components=int(len(comp_infos)),
            total_group_strength=float(np.nansum(np.nan_to_num(score_map[group_mask], nan=0.0))),
        ))

        for comp_idx, (_, comp_mask) in enumerate(comp_infos):
            rec = build_component_record(
                day=day,
                roi_pick_index=roi_pick_index,
                detection_id=detection_id,
                scale_idx=scale_idx,
                scale_name=scale_name,
                N=N,
                S=S,
                period_group_id=gid,
                period_group_center_min=center_min,
                component_idx=comp_idx,
                component_mask=comp_mask,
                score_map=score_map,
                period_map=period_map,
                roi_H=roi_H,
                roi_W=roi_W,
                roi_miny=roi_miny,
                roi_minx=roi_minx,
                H=H,
                W=W,
            )
            detections.append(rec)
            detection_id += 1

    return group_summaries, detections


def cluster_global_period_families(detections, *, period_tol_frac=0.05, period_abs_tol_min=5.0):
    if not detections:
        return [], []

    periods = np.array([
        d["period_min"] if d["period_min"] is not None else np.nan for d in detections
    ], dtype=np.float64)
    weights = np.array([max(1e-12, float(d["strength"])) for d in detections], dtype=np.float64)

    clusters = cluster_period_values(
        periods, weights,
        period_tol_frac=period_tol_frac,
        period_abs_tol_min=period_abs_tol_min,
        max_groups=None,
    )

    family_centers = [float(cl["center"]) for cl in clusters]

    for d in detections:
        p = d["period_min"]
        if p is None or not np.isfinite(float(p)):
            d["family_id"] = None
            d["family_center_min"] = None
            continue

        best_fid = None
        best_dist = np.inf
        for fid, c in enumerate(family_centers):
            if periods_match(
                p, c,
                period_tol_frac=period_tol_frac,
                period_abs_tol_min=period_abs_tol_min,
            ):
                dist = abs(math.log(float(p)) - math.log(float(c)))
                if dist < best_dist:
                    best_dist = dist
                    best_fid = fid

        if best_fid is None:
            d["family_id"] = None
            d["family_center_min"] = None
        else:
            d["family_id"] = int(best_fid)
            d["family_center_min"] = float(family_centers[best_fid])

    family_summaries = []
    for fid, center in enumerate(family_centers):
        fam_det = [d for d in detections if d.get("family_id") == fid]
        if not fam_det:
            continue

        family_summaries.append(dict(
            family_id=int(fid),
            family_center_min=float(center),
            n_components=int(len(fam_det)),
            n_scales=int(len(set(int(d["scale_idx"]) for d in fam_det))),
            total_strength=float(sum(float(d["strength"]) for d in fam_det)),
            scales_present=sorted(set(int(d["N"]) for d in fam_det), reverse=True),
        ))

    family_summaries.sort(key=lambda d: (d["n_scales"], d["total_strength"]), reverse=True)
    return family_centers, family_summaries


# ============================================================
# EVENT CLUSTERING WITHIN PERIOD FAMILIES
# ============================================================
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(int(n)))
        self.rank = [0] * int(n)

    def find(self, x):
        x = int(x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1

    def groups(self):
        out = defaultdict(list)
        for i in range(len(self.parent)):
            out[self.find(i)].append(i)
        return list(out.values())


def _valid_box(box):
    if box is None:
        return False
    if len(box) != 4:
        return False
    if any(v is None for v in box):
        return False
    y0, x0, y1, x1 = box
    vals = np.array([y0, x0, y1, x1], dtype=np.float64)
    if not np.all(np.isfinite(vals)):
        return False
    return (y1 > y0) and (x1 > x0)


def bbox_iou(box_a, box_b):
    """
    box format: (min_y, min_x, max_y, max_x)
    """
    if not (_valid_box(box_a) and _valid_box(box_b)):
        return 0.0

    ay0, ax0, ay1, ax1 = [float(v) for v in box_a]
    by0, bx0, by1, bx1 = [float(v) for v in box_b]

    iy0 = max(ay0, by0)
    ix0 = max(ax0, bx0)
    iy1 = min(ay1, by1)
    ix1 = min(ax1, bx1)

    ih = max(0.0, iy1 - iy0)
    iw = max(0.0, ix1 - ix0)
    inter = ih * iw
    if inter <= 0.0:
        return 0.0

    area_a = max(0.0, ay1 - ay0) * max(0.0, ax1 - ax0)
    area_b = max(0.0, by1 - by0) * max(0.0, bx1 - bx0)
    union = area_a + area_b - inter

    if union <= 0.0:
        return 0.0
    return float(inter / union)


def detection_full_bbox(det):
    box = (
        det.get("bbox_full_min_y"),
        det.get("bbox_full_min_x"),
        det.get("bbox_full_max_y"),
        det.get("bbox_full_max_x"),
    )
    return box if _valid_box(box) else None


def detection_full_centroid(det):
    x = det.get("centroid_full_x", None)
    y = det.get("centroid_full_y", None)
    if x is None or y is None:
        return None
    vals = np.array([x, y], dtype=np.float64)
    if not np.all(np.isfinite(vals)):
        return None
    return float(x), float(y)


def detections_spatially_match(
    d1,
    d2,
    *,
    bbox_iou_thr=0.10,
    centroid_scale_factor=1.5,
    max_scale_idx_gap=None,
    allow_same_scale=False,
):
    """
    Direct edge rule for event clustering.

    Two detections can be linked only if:
      - same family_id
      - optionally not from the same scale
      - optionally close enough in scale index
      - AND (bbox IoU high enough OR centroid distance small enough)
    """
    f1 = d1.get("family_id", None)
    f2 = d2.get("family_id", None)
    if f1 is None or f2 is None or int(f1) != int(f2):
        return False

    s1 = int(d1["scale_idx"])
    s2 = int(d2["scale_idx"])

    if (not allow_same_scale) and (s1 == s2):
        return False

    if max_scale_idx_gap is not None and abs(s1 - s2) > int(max_scale_idx_gap):
        return False

    # bbox overlap test
    box1 = detection_full_bbox(d1)
    box2 = detection_full_bbox(d2)
    iou = bbox_iou(box1, box2)
    iou_ok = iou >= float(bbox_iou_thr)

    # centroid-distance test
    c1 = detection_full_centroid(d1)
    c2 = detection_full_centroid(d2)
    dist_ok = False
    if c1 is not None and c2 is not None:
        dx = float(c1[0] - c2[0])
        dy = float(c1[1] - c2[1])
        dist = math.hypot(dx, dy)
        dist_thr = float(centroid_scale_factor) * max(float(d1["N"]), float(d2["N"]))
        dist_ok = dist <= dist_thr

    return bool(iou_ok or dist_ok)


def build_event_summary(event_id, family_id, members):
    strengths = np.array(
        [max(1e-12, float(d.get("strength", 0.0))) for d in members],
        dtype=np.float64
    )

    periods = np.array(
        [
            np.nan if d.get("period_min", None) is None else float(d["period_min"])
            for d in members
        ],
        dtype=np.float64
    )

    p_ok = np.isfinite(periods) & (periods > 0) & np.isfinite(strengths) & (strengths > 0)
    if np.any(p_ok):
        lp = weighted_median_log(np.log(periods[p_ok]), strengths[p_ok])
        if np.isfinite(lp):
            event_center_min = float(np.exp(lp))
        else:
            event_center_min = float(np.nanmedian(periods[p_ok]))
        event_mean_min = float(np.average(periods[p_ok], weights=strengths[p_ok]))
    else:
        event_center_min = None
        event_mean_min = None

    centroids = []
    centroid_weights = []
    for d in members:
        c = detection_full_centroid(d)
        if c is not None:
            centroids.append(c)
            centroid_weights.append(max(1e-12, float(d.get("strength", 0.0))))

    if centroids:
        centroids = np.asarray(centroids, dtype=np.float64)
        centroid_weights = np.asarray(centroid_weights, dtype=np.float64)
        centroid_full_x = float(np.average(centroids[:, 0], weights=centroid_weights))
        centroid_full_y = float(np.average(centroids[:, 1], weights=centroid_weights))
    else:
        centroid_full_x = None
        centroid_full_y = None

    boxes = [detection_full_bbox(d) for d in members]
    boxes = [b for b in boxes if b is not None]
    if boxes:
        bbox_full = (
            int(min(b[0] for b in boxes)),
            int(min(b[1] for b in boxes)),
            int(max(b[2] for b in boxes)),
            int(max(b[3] for b in boxes)),
        )
    else:
        bbox_full = (None, None, None, None)

    scale_idxs = sorted(set(int(d["scale_idx"]) for d in members))
    scales_present = sorted(set(int(d["N"]) for d in members), reverse=True)
    detection_ids = sorted(int(d["detection_id"]) for d in members)

    return dict(
        event_id=int(event_id),
        family_id=int(family_id),
        family_center_min=(
            None if members[0].get("family_center_min", None) is None
            else float(members[0]["family_center_min"])
        ),
        day=str(members[0]["day"]),
        roi_pick_index=int(members[0]["roi_pick_index"]),
        event_center_min=event_center_min,
        event_mean_min=event_mean_min,
        total_strength=float(np.sum(strengths)),
        n_components=int(len(members)),
        n_scales=int(len(scale_idxs)),
        scale_idxs=scale_idxs,
        scales_present=scales_present,
        detection_ids=detection_ids,
        centroid_full_x=centroid_full_x,
        centroid_full_y=centroid_full_y,
        bbox_full_min_y=bbox_full[0],
        bbox_full_min_x=bbox_full[1],
        bbox_full_max_y=bbox_full[2],
        bbox_full_max_x=bbox_full[3],
    )


def assign_event_ids_within_families(
    detections,
    *,
    bbox_iou_thr=0.10,
    centroid_scale_factor=1.5,
    max_scale_idx_gap=2,
    allow_same_scale=False,
    min_event_scales=4,
):
    """
    Adds:
      - detection['event_id']
      - detection['event_center_min']

    Returns:
      event_summaries
    """
    for d in detections:
        d["event_id"] = None
        d["event_center_min"] = None

    by_family = defaultdict(list)
    for d in detections:
        fid = d.get("family_id", None)
        if fid is None:
            continue
        by_family[int(fid)].append(d)

    event_summaries = []
    next_event_id = 0

    for family_id in sorted(by_family.keys()):
        fam_dets = by_family[family_id]
        n = len(fam_dets)

        uf = UnionFind(n)

        for i in range(n):
            for j in range(i + 1, n):
                if detections_spatially_match(
                    fam_dets[i],
                    fam_dets[j],
                    bbox_iou_thr=float(bbox_iou_thr),
                    centroid_scale_factor=float(centroid_scale_factor),
                    max_scale_idx_gap=max_scale_idx_gap,
                    allow_same_scale=bool(allow_same_scale),
                ):
                    uf.union(i, j)

        groups = uf.groups()

        def _group_sort_key(local_idxs):
            members = [fam_dets[k] for k in local_idxs]
            n_scales = len(set(int(d["scale_idx"]) for d in members))
            total_strength = sum(float(d["strength"]) for d in members)
            return (n_scales, total_strength, len(members))

        groups = sorted(groups, key=_group_sort_key, reverse=True)

        for local_idxs in groups:
            members = [fam_dets[k] for k in local_idxs]
            summary = build_event_summary(
                event_id=next_event_id,
                family_id=family_id,
                members=members,
            )

            # Keep only well-supported events
            if int(summary["n_scales"]) < int(min_event_scales):
                continue

            event_summaries.append(summary)

            for d in members:
                d["event_id"] = int(next_event_id)
                d["event_center_min"] = summary["event_center_min"]

            next_event_id += 1

    event_summaries = sorted(
        event_summaries,
        key=lambda d: (int(d["family_id"]), -int(d["n_scales"]), -float(d["total_strength"]))
    )
    return event_summaries


def summarize_reported_families(detections):
    by_family = defaultdict(list)

    for d in detections:
        fid = d.get("family_id", None)
        if fid is None:
            continue
        by_family[int(fid)].append(d)

    family_summaries = []
    for fid, fam_det in by_family.items():
        center = fam_det[0].get("family_center_min", None)

        family_summaries.append(dict(
            family_id=int(fid),
            family_center_min=(None if center is None else float(center)),
            n_components=int(len(fam_det)),
            n_scales=int(len(set(int(d["scale_idx"]) for d in fam_det))),
            total_strength=float(sum(float(d["strength"]) for d in fam_det)),
            scales_present=sorted(set(int(d["N"]) for d in fam_det), reverse=True),
        ))

    family_summaries.sort(
        key=lambda d: (d["n_scales"], d["total_strength"]),
        reverse=True
    )
    return family_summaries



def write_event_summary_csv(csv_path, event_summaries):
    fields = [
        "event_id", "family_id", "family_center_min",
        "day", "roi_pick_index",
        "event_center_min", "event_mean_min",
        "total_strength", "n_components", "n_scales",
        "centroid_full_x", "centroid_full_y",
        "bbox_full_min_y", "bbox_full_min_x", "bbox_full_max_y", "bbox_full_max_x",
        "scale_idxs", "scales_present", "detection_ids",
    ]

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for rec in event_summaries:
            row = dict(rec)
            row["scale_idxs"] = ",".join(str(x) for x in rec["scale_idxs"])
            row["scales_present"] = ",".join(str(x) for x in rec["scales_present"])
            row["detection_ids"] = ",".join(str(x) for x in rec["detection_ids"])
            writer.writerow(row)

# ============================================================
# PLOTTING
# ============================================================
def plot_full_disk_filaments_bboxes(outpath, image0, mask0, bboxes, *, title):
    fig = plt.figure(figsize=(10, 10))
    plt.imshow(image0, cmap="gray", origin="lower")
    plt.imshow((mask0 > 0).astype(float), cmap="Reds", alpha=0.35, origin="lower")
    for i, (miny, minx, maxy, maxx) in enumerate(bboxes):
        plt.plot([minx, maxx, maxx, minx, minx], [miny, miny, maxy, maxy, miny], "r-", lw=1.0)
        plt.text(
            minx, miny, f"{i}",
            color="yellow", fontsize=8,
            bbox=dict(facecolor="black", alpha=0.4, pad=1.0),
        )
    plt.title(title)
    plt.tight_layout()
    fig.savefig(outpath, dpi=160)
    plt.close(fig)


def plot_cp_calibration(outdir, m, s, qk, delta):
    freqs = CNN_FREQUENCY_GRID.astype(np.float64)
    periods_min = (1.0 / freqs) / 60.0
    factor = np.exp(m + qk * s)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.loglog(periods_min, factor, lw=2)
    ax.set_xlabel("Period (min)")
    ax.set_ylabel("Multiplicative factor")
    ax.set_title(f"Global CP factor ({100*(1-delta):.5f}% upper quantile)")
    ax.invert_xaxis()

    ax = axes[1]
    ax.plot(periods_min, m, label="m")
    ax.plot(periods_min, s, label="s")
    ax.plot(periods_min, qk, label="qk", ls="--")
    ax.set_xlabel("Period (min)")
    ax.set_ylabel("Value")
    ax.set_title("CP residual statistics")
    ax.invert_xaxis()
    ax.legend()

    plt.tight_layout()
    fig.savefig(os.path.join(outdir, "cp_calibration.png"), dpi=150)
    plt.close(fig)


def plot_scale_summary_maps(outdir, scale_name, stack0, union_cov, maps):
    os.makedirs(outdir, exist_ok=True)
    per = maps["best_period_min"]
    z = maps["best_z"]
    sig = maps["sig_any"]
    coh = maps["coherence_logp_std"]

    fig = plt.figure(figsize=(18, 10))
    ax = plt.subplot(231)
    ax.imshow(stack0, cmap="gray", origin="lower")
    ax.set_title(f"{scale_name}: degraded frame[0]")

    ax = plt.subplot(232)
    im = ax.imshow(per, origin="lower")
    ax.set_title(f"{scale_name}: best period (min)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = plt.subplot(233)
    im = ax.imshow(z, origin="lower")
    ax.set_title(f"{scale_name}: best pxx/tau")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = plt.subplot(234)
    ax.imshow(sig, cmap="gray", origin="lower")
    ax.set_title(f"{scale_name}: sig_any")

    ax = plt.subplot(235)
    im = ax.imshow(union_cov, origin="lower")
    ax.set_title(f"{scale_name}: union mask coverage")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = plt.subplot(236)
    im = ax.imshow(coh, origin="lower")
    ax.set_title(f"{scale_name}: coherence std(logP)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    fig.savefig(os.path.join(outdir, f"{scale_name}_scale_maps.png"), dpi=140)
    plt.close(fig)


def plot_scale_period_components(outdir, scale_name, roi_img0, union_mask, detections_in_scale):
    if not detections_in_scale:
        return

    fig = plt.figure(figsize=(10, 8))
    plt.imshow(roi_img0, cmap="gray", origin="lower")
    plt.contour(union_mask.astype(float), levels=[0.5], linewidths=0.8)

    for det in detections_in_scale:
        roi_mask = det["roi_mask"]
        if roi_mask is not None and np.any(roi_mask):
            plt.contour(roi_mask.astype(float), levels=[0.5], linewidths=1.2)
        plt.text(
            det["centroid_roi_x"], det["centroid_roi_y"],
            f"{det['period_min']:.1f}m",
            fontsize=7,
            ha="center", va="center",
            bbox=dict(facecolor="white", alpha=0.6, pad=0.5),
        )

    plt.title(f"{scale_name}: connected components by period")
    plt.tight_layout()
    fig.savefig(os.path.join(outdir, f"{scale_name}_period_components.png"), dpi=150)
    plt.close(fig)


def plot_period_scale_scatter(outpath, detections, *, title):
    if not detections:
        return

    periods = np.array([d["period_min"] for d in detections if d["period_min"] is not None], dtype=np.float64)
    Ns = np.array([d["N"] for d in detections if d["period_min"] is not None], dtype=np.float64)
    strengths = np.array([d["strength"] for d in detections if d["period_min"] is not None], dtype=np.float64)
    fam_ids = np.array([
        -1 if d.get("family_id") is None else int(d["family_id"])
        for d in detections if d["period_min"] is not None
    ], dtype=np.int32)

    if periods.size == 0:
        return

    sizes = 20.0 + 12.0 * np.sqrt(np.maximum(strengths, 0.0))
    color_vals = fam_ids if np.any(fam_ids >= 0) else strengths

    fig = plt.figure(figsize=(10, 7))
    sc = plt.scatter(periods, Ns, s=sizes, c=color_vals, alpha=0.8)
    plt.xscale("log")
    plt.xlabel("Period (min)")
    plt.ylabel("Scale N")
    plt.title(title)
    plt.grid(True, which="both", alpha=0.25)

    cbar = plt.colorbar(sc)
    if np.any(fam_ids >= 0):
        cbar.set_label("Family ID")
    else:
        cbar.set_label("Strength")

    plt.tight_layout()
    fig.savefig(outpath, dpi=150)
    plt.close(fig)


def plot_period_family_spatial_maps(outdir, roi_img0, union_mask, scales, detections, family_summaries):
    os.makedirs(outdir, exist_ok=True)
    if not detections or not family_summaries:
        return

    scale_name_by_idx = {i: sc["name"] for i, sc in enumerate(scales)}

    for fam in family_summaries:
        fid = fam["family_id"]
        fam_center = fam["family_center_min"]
        fam_dets = [d for d in detections if d.get("family_id") == fid]
        if not fam_dets:
            continue

        dets_by_scale = defaultdict(list)
        for d in fam_dets:
            dets_by_scale[int(d["scale_idx"])].append(d)

        scale_idxs = sorted(dets_by_scale.keys())
        n_panels = len(scale_idxs)
        ncols = min(3, n_panels)
        nrows = int(math.ceil(n_panels / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)
        axes = axes.ravel()

        for ax in axes[n_panels:]:
            ax.axis("off")

        for ax, sidx in zip(axes[:n_panels], scale_idxs):
            ax.imshow(roi_img0, cmap="gray", origin="lower")
            ax.contour(union_mask.astype(float), levels=[0.5], linewidths=0.8)

            for det in dets_by_scale[sidx]:
                roi_mask = det["roi_mask"]
                if roi_mask is not None and np.any(roi_mask):
                    ax.contour(roi_mask.astype(float), levels=[0.5], linewidths=1.5)
                ax.plot(det["centroid_roi_x"], det["centroid_roi_y"], marker="x", ms=8, mew=1.5)
                ax.text(
                    det["centroid_roi_x"], det["centroid_roi_y"],
                    f"{det['period_min']:.1f}m",
                    fontsize=7,
                    ha="left", va="bottom",
                    bbox=dict(facecolor="white", alpha=0.6, pad=0.5),
                )

            ax.set_title(f"{scale_name_by_idx[sidx]} | {len(dets_by_scale[sidx])} component(s)")

        fig.suptitle(
            f"Family {fid} | center ≈ {fam_center:.2f} min | "
            f"{fam['n_scales']} scale(s), {fam['n_components']} component(s)"
        )
        plt.tight_layout()
        fig.savefig(os.path.join(outdir, f"family_{fid:03d}_P{fam_center:.2f}min.png"), dpi=150)
        plt.close(fig)


# ============================================================
# WRITERS
# ============================================================
def _strip_masks_for_serialization(detections):
    out = []
    for d in detections:
        dd = dict(d)
        dd.pop("roi_mask", None)
        out.append(dd)
    return out


def write_component_csv(csv_path, detections):
    fields = [
        "detection_id", "day", "roi_pick_index",
        "scale_idx", "scale_name", "N", "S",
        "period_group_id", "period_group_center_min", "component_idx",
        "period_min", "period_mean_min",
        "strength", "area_degpx",
        "family_id", "family_center_min",
        "event_id", "event_center_min",
        "centroid_deg_i", "centroid_deg_j",
        "centroid_roi_x", "centroid_roi_y",
        "centroid_full_x", "centroid_full_y",
        "bbox_deg_min_i", "bbox_deg_min_j", "bbox_deg_max_i", "bbox_deg_max_j",
        "bbox_roi_min_y", "bbox_roi_min_x", "bbox_roi_max_y", "bbox_roi_max_x",
        "bbox_full_min_y", "bbox_full_min_x", "bbox_full_max_y", "bbox_full_max_x",
    ]

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for d in detections:
            row = {k: d.get(k, None) for k in fields}
            writer.writerow(row)


def write_json(json_path, payload):
    with open(json_path, "w") as f:
        json.dump(
            payload, f, indent=2,
            default=lambda x: None if (isinstance(x, float) and math.isnan(x)) else x
        )


# ============================================================
# MAIN PIPELINE
# ============================================================
def run_once(
    *,
    day,
    data_h5,
    masks_h5,
    outdir,
    cnn_weights_path,
    cp_n_calib=100_000,
    cp_delta=0.00001,
    cp_batch_size_cnn=4096,
    detect_fmin_hz=DETECTION_FMIN_HZ,
    detect_fmax_hz=DETECTION_FMAX_HZ,
    roi_pick="largest",
    roi_pick_index=0,
    roi_min_area=2000,
    roi_pad=15,
    union_min_frames=6 * 60,
    union_dilate_iter=2,
    cov_thr=0.30,
    top_m_peaks=4,
    peak_min_prom_frac=0.05,
    scales_mode="distinct_quotients",
    Nmax=51,
    Nmin=20,
    n_scales=12,
    overlap_frac=0.75,
    period_tol_frac=0.05,
    period_abs_tol_min=5.0,
    min_component_area=1,
    max_period_groups_per_scale=None,
    max_components_per_group=None,
    null_mode="none",
    null_seed=0,
    n_jobs=N_PIXEL_WORKERS,
    cp_cache_path=None,
):
    os.makedirs(outdir, exist_ok=True)

    plots_out = os.path.join(outdir, "plots")
    tables_out = os.path.join(outdir, "tables")
    families_out = os.path.join(outdir, "period_families")
    os.makedirs(plots_out, exist_ok=True)
    os.makedirs(tables_out, exist_ok=True)
    os.makedirs(families_out, exist_ok=True)

    print(f"\n[run] day={day}  roi_pick_index={roi_pick_index}  outdir={outdir}")

    with h5.File(data_h5, "r") as hf:
        images = np.array(hf["time_series"][:], dtype=np.float32)
        tdeltas = np.array(hf["tdeltas"][:], dtype=np.float32)

    with h5.File(masks_h5, "r") as hm:
        masks = np.array(hm["masks"][:], dtype=np.uint8)

    order = np.argsort(tdeltas)
    tdeltas = tdeltas[order]
    images = images[order]
    masks = masks[order]

    H, W = images.shape[1], images.shape[2]
    dt_med = np.median(np.diff(tdeltas[np.isfinite(tdeltas)]))
    print(f"[data] images={images.shape}  dt_med={dt_med/60:.2f} min")

    cnn_model = load_cnn(cnn_weights_path)
    scaler = _build_scaler()

    if cp_cache_path is None:
        cp_cache_path = get_cp_cache_path(
            os.path.abspath(os.path.join(outdir, os.pardir)),
            day,
            cp_delta
        )

    if not os.path.exists(cp_cache_path):
        raise FileNotFoundError(f"Expected precomputed CP cache was not found: {cp_cache_path}")

    print(f"[CP] Loading cached calibration stats from {cp_cache_path}")
    _cpdata = np.load(cp_cache_path)
    m_cp = _cpdata["m"]
    s_cp = _cpdata["s"]
    qk = _cpdata["qk"]

    if not (np.all(np.isfinite(m_cp)) and np.all(np.isfinite(s_cp)) and np.all(np.isfinite(qk))):
        raise RuntimeError(
            f"Loaded CP cache contains non-finite values: {cp_cache_path}. "
            "Delete it and rebuild calibration."
        )

    plot_cp_calibration(plots_out, m_cp, s_cp, qk, delta=float(cp_delta))

    day_root = os.path.abspath(os.path.join(outdir, os.pardir))
    os.makedirs(day_root, exist_ok=True)
    day_overview_png = os.path.join(day_root, "full_disk_filaments_bboxes.png")
    if not os.path.exists(day_overview_png):
        regs_sorted = list_candidate_regions_from_mask(masks[0], min_area=int(roi_min_area))
        plot_full_disk_filaments_bboxes(
            day_overview_png, images[0], masks[0],
            [r.bbox for r in regs_sorted],
            title=f"{day} full disk: filaments + bbox indices (area desc)\nroi_min_area={roi_min_area}"
        )

    bbox = expand_bbox(
        select_roi_bbox_from_first_mask(
            masks[0],
            min_area=roi_min_area,
            pick=roi_pick,
            pick_index=roi_pick_index,
        ),
        pad=roi_pad, H=H, W=W
    )
    roi_miny, roi_minx, roi_maxy, roi_maxx = bbox
    roi_H = roi_maxy - roi_miny
    roi_W = roi_maxx - roi_minx

    ROI_images = images[:, roi_miny:roi_maxy, roi_minx:roi_maxx]
    ROI_masks = masks[:, roi_miny:roi_maxy, roi_minx:roi_maxx]

    union_mask = (ROI_masks > 0).sum(axis=0) >= int(union_min_frames)
    if union_dilate_iter and union_dilate_iter > 0:
        union_mask = binary_dilation(union_mask, iterations=int(union_dilate_iter))

    fig = plt.figure(figsize=(8, 6))
    plt.imshow(ROI_images[0], cmap="gray", origin="lower")
    plt.contour(union_mask.astype(float), levels=[0.5], linewidths=1.2)
    plt.title("ROI frame[0] + persistent union mask contour")
    plt.tight_layout()
    fig.savefig(os.path.join(plots_out, "roi_union_mask.png"), dpi=140)
    plt.close(fig)

    if scales_mode == "distinct_quotients":
        Ns = choose_scales_distinct_quotients(roi_H, start=int(Nmax), stop=int(Nmin) - 1)
    elif scales_mode == "log_ladder":
        Ns = choose_scales_log_ladder(int(Nmax), int(Nmin), n=int(n_scales))
    else:
        raise ValueError("scales_mode must be distinct_quotients or log_ladder")

    scales = [
        dict(
            N=int(N),
            S=max(1, round(N * (1 - overlap_frac))),
            name=f"N{int(N)}_S{max(1, round(N * (1 - overlap_frac)))}"
        )
        for N in Ns
    ]
    print(f"[scales] {len(scales)}: " + ", ".join(sc["name"] for sc in scales))

    rng = np.random.default_rng(int(null_seed))

    all_group_summaries = []
    all_detections = []
    per_scale_cache = []
    next_detection_id = 0

    for scale_idx, sc in enumerate(scales):
        N, S, name = sc["N"], sc["S"], sc["name"]
        print(f"\n[scale {scale_idx+1}/{len(scales)}] {name}")

        stack = degrade_stack_unweighted(ROI_images, N=N, S=S)
        stack -= np.nanmean(stack, axis=0, keepdims=True)
        stack = apply_null_time_transform(stack, null_mode, rng)

        cov = degrade_mask_coverage(union_mask, N=N, S=S)
        keep_mask = (cov >= float(cov_thr))

        maps = analyze_degraded_stack_multipeak_cp(
            tdeltas, stack,
            cnn_model, scaler,
            m_cp, s_cp, qk,
            keep_mask=keep_mask,
            top_m_peaks=int(top_m_peaks),
            peak_min_prom_frac=float(peak_min_prom_frac),
            n_jobs=int(n_jobs),
            batch_size_cnn=int(cp_batch_size_cnn),
            detect_fmin_hz=float(detect_fmin_hz),
            detect_fmax_hz=float(detect_fmax_hz),
        )

        n_sig = int((maps["sig_any"] & maps["keep_mask"]).sum())
        print(f"[scale] sig pixels={n_sig}")

        plot_scale_summary_maps(plots_out, name, stack0=stack[0], union_cov=cov, maps=maps)

        group_summaries, detections = extract_scale_period_components(
            day=day,
            roi_pick_index=roi_pick_index,
            scale_idx=scale_idx,
            scale_name=name,
            N=N,
            S=S,
            maps=maps,
            roi_H=roi_H,
            roi_W=roi_W,
            roi_miny=roi_miny,
            roi_minx=roi_minx,
            H=H,
            W=W,
            period_tol_frac=float(period_tol_frac),
            period_abs_tol_min=float(period_abs_tol_min),
            min_component_area=int(min_component_area),
            max_period_groups=max_period_groups_per_scale,
            max_components_per_group=max_components_per_group,
            base_detection_id=next_detection_id,
        )

        print(f"[scale] period groups={len(group_summaries)}  components={len(detections)}")

        detections_sorted = sorted(
            detections,
            key=lambda d: (float(d["period_min"]) if d["period_min"] is not None else np.inf, -float(d["strength"]))
        )
        plot_scale_period_components(
            plots_out, name, ROI_images[0], union_mask, detections_sorted
        )

        scale_csv = os.path.join(tables_out, f"{name}_components.csv")
        scale_json = os.path.join(tables_out, f"{name}_components.json")
        write_component_csv(scale_csv, _strip_masks_for_serialization(detections))
        write_json(scale_json, dict(
            day=str(day),
            roi_pick_index=int(roi_pick_index),
            scale_idx=int(scale_idx),
            scale_name=str(name),
            N=int(N),
            S=int(S),
            n_groups=int(len(group_summaries)),
            n_components=int(len(detections)),
            groups=group_summaries,
            components=_strip_masks_for_serialization(detections),
        ))

        all_group_summaries.extend(group_summaries)
        all_detections.extend(detections)
        next_detection_id += len(detections)
        per_scale_cache.append(dict(scale_idx=scale_idx, scale_name=name, N=N, S=S, maps=maps))

    if not all_detections:
        print("[done] No connected components found.")
        return

    _, _all_family_summaries = cluster_global_period_families(
        all_detections,
        period_tol_frac=float(period_tol_frac),
        period_abs_tol_min=float(period_abs_tol_min),
    )

    event_summaries = assign_event_ids_within_families(
        all_detections,
        bbox_iou_thr=0.05,
        centroid_scale_factor=2,
        max_scale_idx_gap=3,
        allow_same_scale=False,
        min_event_scales=3,
    )

    # Keep only detections that belong to events seen in >= 3 scales
    all_detections = [d for d in all_detections if d.get("event_id") is not None]

    if not all_detections:
        print("[done] No events survived the min_event_scales >= 3 filter.")
        return

    family_summaries = summarize_reported_families(all_detections)

    plot_period_scale_scatter(
        os.path.join(plots_out, "period_vs_scale_components.png"),
        all_detections,
        title="Connected-component detections: period vs scale",
    )

    plot_period_family_spatial_maps(
        families_out,
        ROI_images[0],
        union_mask,
        scales,
        all_detections,
        family_summaries,
    )

    all_components_csv = os.path.join(tables_out, "all_components.csv")
    all_components_json = os.path.join(tables_out, "all_components.json")
    family_summary_json = os.path.join(tables_out, "period_families.json")
    family_summary_csv = os.path.join(tables_out, "period_families.csv")
    event_summary_json = os.path.join(tables_out, "events.json")
    event_summary_csv = os.path.join(tables_out, "events.csv")

    write_component_csv(all_components_csv, _strip_masks_for_serialization(all_detections))
    write_json(all_components_json, dict(
        day=str(day),
        roi_pick=str(roi_pick),
        roi_pick_index=int(roi_pick_index),
        roi_bbox=dict(
            min_y=int(roi_miny), min_x=int(roi_minx),
            max_y=int(roi_maxy), max_x=int(roi_maxx),
        ),
        cp_delta=float(cp_delta),
        n_scales=int(len(scales)),
        n_components=int(len(all_detections)),
        n_families=int(len(family_summaries)),
        n_events=int(len(event_summaries)),
        components=_strip_masks_for_serialization(all_detections),
    ))

    write_json(family_summary_json, dict(
        day=str(day),
        roi_pick_index=int(roi_pick_index),
        n_families=int(len(family_summaries)),
        families=family_summaries,
    ))

    write_json(event_summary_json, dict(
        day=str(day),
        roi_pick_index=int(roi_pick_index),
        n_events=int(len(event_summaries)),
        events=event_summaries,
    ))

    write_event_summary_csv(event_summary_csv, event_summaries)

    with open(family_summary_csv, "w", newline="") as f:
        fields = [
            "family_id", "family_center_min", "n_components",
            "n_scales", "total_strength", "scales_present",
        ]
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for rec in family_summaries:
            row = dict(rec)
            row["scales_present"] = ",".join(str(x) for x in rec["scales_present"])
            writer.writerow(row)

    summary_txt = os.path.join(tables_out, "summary.txt")
    with open(summary_txt, "w") as f:
        f.write("CONNECTED-COMPONENT PERIOD CATALOG\n")
        f.write("=================================\n\n")
        f.write(f"day: {day}\n")
        f.write(f"roi_pick_index: {roi_pick_index}\n")
        f.write(f"cp_delta: {cp_delta}\n")
        f.write(f"n_scales: {len(scales)}\n")
        f.write(f"n_components: {len(all_detections)}\n")
        f.write(f"n_families: {len(family_summaries)}\n")
        f.write(f"n_events: {len(event_summaries)}\n\n")

        f.write("Families:\n")
        for fam in family_summaries:
            f.write(
                f"  family_id={fam['family_id']:>3d}  "
                f"center={fam['family_center_min']:.3f} min  "
                f"n_scales={fam['n_scales']:>2d}  "
                f"n_components={fam['n_components']:>3d}  "
                f"strength={fam['total_strength']:.2f}\n"
            )

    print(
        f"[done] {len(all_detections)} component(s), "
        f"{len(family_summaries)} family/families, "
        f"{len(event_summaries)} event(s) -> {tables_out}"
    )


# ============================================================
# ENTRY POINT
# ============================================================
def main(day: str = None, index: int = None, cp_cache_path: str = None):
    day = "20140101" if day is None else day
    data_h5 = f"../data/{day}/updated/{day}_data_modified.h5"
    masks_h5 = f"../data/{day}/updated/{day}_masks.h5"

    roi_pick = "index"
    roi_pick_index = index if index is not None else 0
    roi_min_area = 750
    roi_pad = 25
    outdir = f"./period_component_results_v2/{day}/{roi_pick_index}"

    run_once(
        day=day,
        data_h5=data_h5,
        masks_h5=masks_h5,
        outdir=outdir,
        cnn_weights_path="../CNN/BestFit/BestFitWeights.h5",
        cp_n_calib=1_000_000,
        cp_delta=CP_DELTA,
        cp_batch_size_cnn=4096,
        detect_fmin_hz=1.0 / (3.0 * 3600.0),
        detect_fmax_hz=1.0e-3,
        roi_pick=roi_pick,
        roi_pick_index=roi_pick_index,
        roi_min_area=roi_min_area,
        roi_pad=roi_pad,
        union_min_frames=6 * 60,
        union_dilate_iter=2,
        cov_thr=0.30,
        top_m_peaks=4,
        peak_min_prom_frac=0.10,
        scales_mode="distinct_quotients",
        Nmax=71,
        Nmin=9,
        n_scales=12,
        overlap_frac=0.75,
        period_tol_frac=0.05,
        period_abs_tol_min=5.0,
        min_component_area=1,
        max_period_groups_per_scale=None,
        max_components_per_group=None,
        null_mode="none",
        null_seed=0,
        n_jobs=N_PIXEL_WORKERS,
        cp_cache_path=cp_cache_path,
    )


if __name__ == "__main__":
    days = sorted(os.listdir("../data/"))

    for day in days:
        data_h5 = f"../data/{day}/updated/{day}_data_modified.h5"
        masks_h5 = f"../data/{day}/updated/{day}_masks.h5"

        with h5.File(masks_h5, "r") as hf:
            masks = np.array(hf["masks"][:], dtype=np.uint8)

        n_filaments = len(list_candidate_regions_from_mask(masks[0], min_area=750))
        print(f"\n=== Day {day}: {n_filaments} filaments ===")

        cp_cache_path = ensure_cp_cache_for_day(
            day=day,
            data_h5=data_h5,
            masks_h5=masks_h5,
            outroot=f"./period_component_results_v2/{day}",
            cnn_weights_path="../CNN/BestFit/BestFitWeights.h5",
            cp_n_calib=1_000_000,
            cp_delta=CP_DELTA,
            cp_batch_size_cnn=4096,
            n_jobs=N_PIXEL_WORKERS,
        )

        Parallel(n_jobs=N_FILAMENT_WORKERS, verbose=5)(
            delayed(main)(day=day, index=i, cp_cache_path=cp_cache_path)
            for i in range(n_filaments)
        )